# CO2Dot calibration run (multi-device, LI-6800)

Generates the dataset for refitting the two-state indicator model: end-member channel
vectors E0/E1, per-device scale, alpha_ref(a_w), dH(a_w) and the dye time constant.
All logic lives in `co2dot_calib.py`; this notebook only configures and starts it.

**Architecture.** A background *logger* reads the LI-6800 and every CO2Dot at a fixed
cadence and appends one JSON line per cycle to `cycles.jsonl`. The *scheduler* walks the
protocol (RH blocks > T levels > CO2 steps), changes setpoints and phase tags only, and
writes `events.jsonl`. Everything is checkpointed: Ctrl-C stops cleanly and `run.run()`
resumes. `load_run()` turns a run folder into tidy DataFrames.

**Before starting**
1. Devices in the chamber facing the air stream; LI-6800 leaf thermocouple taped to one film.
2. Fresh CO2 cartridge, soda lime, desiccant and humidifier columns checked (30 h run).
3. Device identity = USB serial number (chip MAC). Fill `LABELS` below once per device.
4. Preflight must show a real LED signal (diff ≫ dark) and no saturation on any device.

In [ ]:
import time
from datetime import datetime
from pathlib import Path
from co2dot_calib import (connect_dots, apply_settings, preflight, Protocol, Run, LicorLive,
                          MockLicor, mock_dots, load_run, settled_points)

## 1. Devices

In [ ]:
LABELS = {                      # device_id (USB serial = MAC) -> bench label; extend as devices are added
    "3C:DC:75:0D:FA:64": "A",
}
dots = connect_dots(labels=LABELS)
settings = apply_settings(dots, gain=5, atime=100, astep=999)     # common settings; use per_device={...} to override
report = preflight(dots, led_ma=10, n=3)

## 2. LI-6800

In [ ]:
from li_mqtt import LI6800
li = LI6800.connect()
print("broker", li.broker)
time.sleep(2)
print({k: li.get(k) for k in ("CO2_s", "CO2_r", "H2O_s", "Tchamber", "Txchg", "Tleaf", "RHcham", "Flow", "Press")})
licor = LicorLive(li)

## 3. Protocol

Edit the grid and timings here. `rh_cap_by_t` is the static condensation guard (RH 75 % only
at 27 and 32 °C); the live guard uses the dew point of the sample air against the exchanger
and board temperatures. `cartridge_life_h` is an assumption: update it from experience.

In [ ]:
P = Protocol(
    rh_levels=(30, 45, 60, 75),
    t_levels=(17, 22, 27, 32),
    co2_levels=(0, 150, 300, 400, 500, 700, 1000, 2000),
    rh_cap_by_t={17: 60, 22: 60},
    co2_dwell_s=300, co2_dwell_long_s=600,
    rh_settle_min_s=45 * 60, rh_settle_max_s=90 * 60,
    t_settle_min_s=15 * 60,
    passes=2,
    flow_umol_s=400,
    cartridge_life_h=10,
)
print(P.feasibility_table())
est = P.estimate()
print(f"\n{est['cells']} cells, {est['co2_steps']} CO2 steps, {est['skipped']} skipped;  ~{est['hours']} h total, "
      f"mixer active ~{est['mixer_hours']} h  = {est['cartridges']:.1f} cartridges of {P.cartridge_life_h} h")

## 4. Dry run without hardware (optional, ~2 min)

Validates the schedule, the file format and the loader with mock instruments and a
compressed protocol. Output goes to `co2dot_data/<timestamp>_mock_dryrun`.

In [ ]:
# lic = MockLicor(tau_s={"co2": 2, "t": 2, "rh": 2})
# dry = Run(Path("co2dot_data") / (datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + "_mock_dryrun"),
#           mock_dots(2, lic), lic, Protocol.quick(), cadence_s=1.0)
# dry.run()
# cyc, ev, meta = load_run(dry.outdir); print(cyc.shape); print(ev["kind"].value_counts())

## 5. Run

Starts the logger and the scheduler. Ctrl-C (interrupt the kernel) stops cleanly: the chamber is
returned to ambient, the checkpoint is kept. Run the **resume** cell to continue. If the run
returns `'paused'`, the CO2 cartridge is probably empty: replace it and resume.

In [ ]:
RUN_TAG = "calib"
run = Run(Path("co2dot_data") / (datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + f"_{RUN_TAG}"),
          dots, licor, P, cadence_s=10.0, led_ma=10,
          meta_extra={"operator": "", "notes": "devices in chamber facing the air stream; thermocouple on film A"})
print("run folder:", run.outdir)
status = run.run()
print("status:", status)

In [ ]:
# Resume after Ctrl-C / cartridge swap (same run object -> same folder and checkpoint):
# status = run.run(); print(status)

## 6. Quick look

In [ ]:
import matplotlib.pyplot as plt
cyc, ev, meta = load_run(run.outdir)
print(cyc.shape, "cycles x devices;", len(ev), "events")
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for lab, g in cyc.groupby("label"):
    ax[0].plot(g["ts"], g["lr_630_515"], lw=0.8, label=lab)
    ax[2].plot(g["ts"], g["env_T"], lw=0.8, label=f"{lab} board T"); ax[2].plot(g["ts"], g["env_RH"], lw=0.8, ls=":", label=f"{lab} board RH")
g0 = cyc.drop_duplicates("cycle")
ax[1].plot(g0["ts"], g0["licor_CO2_s"], "k", lw=0.8); ax[1].set_ylabel("CO2_s (ppm)")
ax[2].plot(g0["ts"], g0["licor_Tchamber"], "k", lw=0.8, label="Tchamber"); ax[2].plot(g0["ts"], g0["licor_RHcham"], "k:", lw=0.8, label="RHcham")
ax[0].set_ylabel("log10(630/515)"); ax[0].legend(ncol=4, fontsize=8); ax[2].legend(ncol=4, fontsize=8)
plt.tight_layout()
sp = settled_points(cyc, last_s=120)
sp[["phase_step", "label", "kind", "pass_", "t_set", "rh_set", "co2_set", "n", "licor_CO2_s", "env_T", "env_RH", "lr_630_515"]].round(3)

## 7. Close

In [ ]:
run.close()        # stops the logger (if still running) and closes every CO2Dot port
li.close()